# GNN 사기 탐지 — 04. Baseline: Review-centric RGCN

## 설계 목적: 동질 그래프의 한계 증명

| 항목 | 설계 |
|------|------|
| **노드** | Review 1종 (78차원) — User / Restaurant 노드 없음 |
| **엣지** | R-S-R (같은 식당+별점) + R-Sim-R (텍스트 유사도) = 2개 관계 |
| **모델** | 2-Layer RGCN (Relational GCN) |
| **한계** | 유저 행동 맥락 없음 → 동일 유저 공모 패턴 직접 탐지 불가 |

결과는 `results_04.json`으로 저장 → 05/06 모델과 3-way 비교.

In [1]:
# [00] 임포트 및 환경 설정
import os, sys, time, warnings
import numpy as np
import pandas as pd
import json as _json
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, average_precision_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import to_undirected

BASE_DIR = os.getcwd()
os.chdir(BASE_DIR)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ 환경 설정 완료 | 디바이스: {device}')
print(f'   PyTorch: {torch.__version__}')

✅ 환경 설정 완료 | 디바이스: cpu
   PyTorch: 2.11.0+cpu


In [2]:
# [01] 하이퍼파라미터
CFG = {
    'test_size'         : 0.20,
    'val_size'          : 0.20,
    'random_state'      : 42,
    'tfidf_max_features': 5000,
    'svd_n_components'  : 64,
    'rsr_max_per_node'  : 10,
    'rsim_thr'          : 0.70,
    'hidden_dim'        : 128,
    'dropout'           : 0.30,
    'lr'                : 1e-3,
    'wd'                : 1e-4,
    'epochs'            : 100,
    'patience'          : 10,
}
print('✅ 설정 완료')
print(f'   관계 수: 2  (R-S-R / R-Sim-R)')
print(f'   입력 차원 목표: 78  (14 수치 + 64 TF-IDF SVD)')

✅ 설정 완료
   관계 수: 2  (R-S-R / R-Sim-R)
   입력 차원 목표: 78  (14 수치 + 64 TF-IDF SVD)


In [3]:
# [02] 데이터 로드
print('데이터 로드 중...')
df = pd.read_parquet('yelpzip_sampled.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)  # 시간 순 정렬 (분리 기준)

assert set(df['label'].unique()).issubset({0, 1}), '라벨 변환 필요!'
n_total = len(df)
n_fraud  = (df['label'] == 1).sum()
n_normal = (df['label'] == 0).sum()
print(f'✅ 데이터 로드: {n_total:,}건')
print(f'   사기(1): {n_fraud:,} ({100*n_fraud/n_total:.1f}%)')
print(f'   정상(0): {n_normal:,} ({100*n_normal/n_total:.1f}%)')
print(f'   유저: {df["user_id"].nunique():,}명  |식당: {df["prod_id"].nunique():,}개')
print(f'   기간: {df["date"].min().date()} ~ {df["date"].max().date()}')

데이터 로드 중...
✅ 데이터 로드: 50,950건
   사기(1): 12,659 (24.8%)
   정상(0): 38,291 (75.2%)
   유저: 17,439명  |식당: 200개
   기간: 2013-01-01 ~ 2014-12-31


In [4]:
# [03] 텍스트 피처 추출 (7개)
MENU_WORDS = {
    'chicken','beef','pizza','salad','pasta','fish','shrimp','steak',
    'burger','sandwich','soup','rice','noodles','sushi','tacos','bread',
    'cake','dessert','cheese','bacon','turkey','salmon','lobster','crab',
    'pork','lamb','duck','tofu','mushroom','fries','wings','ribs',
    'brisket','sausage','waffle','pancake','curry','ramen','pho',
    'gyoza','tempura','dumplings','kebab','burrito','nachos'
}
POS_WORDS = {
    'great','amazing','delicious','excellent','wonderful','fantastic',
    'outstanding','superb','brilliant','perfect','awesome','incredible',
    'lovely','magnificent','terrific','fabulous'
}
NEG_WORDS = {
    'bad','terrible','awful','horrible','disgusting','dreadful','poor',
    'disappointing','mediocre','tasteless','bland','rude','slow',
    'dirty','overpriced','waste','worst'
}

def extract_text_features(texts):
    rows = []
    for text in texts:
        text = str(text)
        words = text.split()
        n = max(len(words), 1)
        chars = max(len(text), 1)
        unique_words = set(w.lower().strip('.,!?') for w in words)
        pos  = sum(1 for w in words if w.lower().strip('.,!?') in POS_WORDS)
        neg  = sum(1 for w in words if w.lower().strip('.,!?') in NEG_WORDS)
        menu = sum(1 for w in words if w.lower().strip('.,!?') in MENU_WORDS)
        rows.append({
            'text_len'     : len(text),
            'word_count'   : len(words),
            'digit_ratio'  : sum(c.isdigit() for c in text) / chars,
            'menu_mentions': float(menu),
            'special_ratio': sum(not c.isalnum() and not c.isspace() for c in text) / chars,
            'ttr'          : len(unique_words) / n,
            'sentiment'    : (pos - neg) / n,
        })
    return pd.DataFrame(rows)

print('텍스트 피처 추출 중...')
t0 = time.time()
text_feats = extract_text_features(df['text'].values)
print(f'✅ 완료 ({time.time()-t0:.0f}s) | shape: {text_feats.shape}')

텍스트 피처 추출 중...
✅ 완료 (9s) | shape: (50950, 7)


In [5]:
# [04] Train / Val / Test 분리 (시간 기준 Option B — 사기 비율 균형 cutoff 탐색)
N      = len(df)
labels = df['label'].values

best_idx, best_diff = None, float('inf')
for cutoff in range(int(N * 0.72), int(N * 0.88)):
    tr_ratio = labels[:cutoff].mean()
    te_ratio = labels[cutoff:].mean()
    diff = abs(tr_ratio - te_ratio)
    if diff < best_diff:
        best_diff, best_idx = diff, cutoff

idx_trainval = np.arange(best_idx)
idx_test     = np.arange(best_idx, N)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=CFG['val_size'],
    random_state=CFG['random_state'],
    stratify=labels[idx_trainval]
)

train_mask = torch.zeros(N, dtype=torch.bool); train_mask[idx_train] = True
val_mask   = torch.zeros(N, dtype=torch.bool); val_mask[idx_val]     = True
test_mask  = torch.zeros(N, dtype=torch.bool); test_mask[idx_test]   = True

cutoff_date = df.iloc[best_idx]['date'].date()
print(f'시간 분리 기준일: {cutoff_date}  (사기 비율 차이: {best_diff:.4f})')
print(f'Train: {len(idx_train):,} ({len(idx_train)/N*100:.1f}%)  '
      f'사기 비율: {labels[idx_train].mean():.3f}')
print(f'Val  : {len(idx_val):,} ({len(idx_val)/N*100:.1f}%)  '
      f'사기 비율: {labels[idx_val].mean():.3f}')
print(f'Test : {len(idx_test):,} ({len(idx_test)/N*100:.1f}%)  '
      f'사기 비율: {labels[idx_test].mean():.3f}')

시간 분리 기준일: 2014-06-28  (사기 비율 차이: 0.0467)
Train: 29,392 (57.7%)  사기 비율: 0.235
Val  : 7,348 (14.4%)  사기 비율: 0.235
Test : 14,210 (27.9%)  사기 비율: 0.282


In [6]:
# [05] TF-IDF + SVD 텍스트 임베딩  (훈련 데이터로만 fit — 데이터 누수 방지)
print('TF-IDF + SVD 임베딩 중...')
t0 = time.time()
tfidf = TfidfVectorizer(
    max_features=CFG['tfidf_max_features'],
    ngram_range=(1, 2), min_df=3, max_df=0.95,
    strip_accents='unicode', sublinear_tf=True
)
# 훈련 텍스트로만 어휘·IDF 학습
tfidf.fit(df['text'].iloc[idx_train].astype(str))
tfidf_mat  = tfidf.transform(df['text'].astype(str))   # 전체 transform

svd = TruncatedSVD(n_components=CFG['svd_n_components'], random_state=CFG['random_state'])
# 훈련 행렬로만 SVD 학습
svd.fit(tfidf_mat[idx_train])
text_embed = svd.transform(tfidf_mat)                  # 전체 transform
print(f'✅ 완료 ({time.time()-t0:.0f}s) | shape: {text_embed.shape}')
print(f'   설명 분산: {svd.explained_variance_ratio_.sum():.3f}')

TF-IDF + SVD 임베딩 중...
✅ 완료 (34s) | shape: (50950, 64)
   설명 분산: 0.128


In [7]:
# [06] 피처 행렬 조립 (78차원 - Review 고유 피처만)
df_feat = df[['user_id','prod_id','rating','date','label']].copy().reset_index(drop=True)
df_feat['weekday'] = df['date'].dt.weekday.astype(float)
df_feat['month']   = df['date'].dt.month.astype(float)

# features_rating.csv: rating_deviation, is_cold_start, is_single_review
fr = pd.read_csv('features_rating.csv', index_col=0)
fr_dedup = fr[['user_id','prod_id','rating_deviation','is_cold_start','is_single_review']]\
             .drop_duplicates(subset=['user_id','prod_id'])
df_feat = df_feat.merge(fr_dedup, on=['user_id','prod_id'], how='left')

# 텍스트 피처 추가
for col in text_feats.columns:
    df_feat[col] = text_feats[col].values

# rpr_suspicion 분위수: 훈련 데이터(idx_train)만 사용 — 누수 방지
tl_35 = np.nanpercentile(df_feat['text_len'].values[idx_train], 35)
st_65 = np.nanpercentile(df_feat['sentiment'].values[idx_train], 65)
df_feat['rpr_suspicion'] = (
    (df_feat['text_len']      <= tl_35).astype(float) +
    (df_feat['menu_mentions'] == 0).astype(float) +
    (df_feat['sentiment']     >= st_65).astype(float)
) / 3.0

df_feat.fillna(0, inplace=True)

NUM_COLS = [
    'rating', 'weekday', 'month',
    'rating_deviation', 'is_cold_start', 'is_single_review',
    'rpr_suspicion',
    'text_len', 'word_count', 'digit_ratio', 'menu_mentions',
    'special_ratio', 'ttr', 'sentiment',
]
print(f'수치 피처: {len(NUM_COLS)}차원')
print(f'TF-IDF SVD: {CFG["svd_n_components"]}차원')
print(f'합계: {len(NUM_COLS) + CFG["svd_n_components"]}차원  (목표: 78)')

수치 피처: 14차원
TF-IDF SVD: 64차원
합계: 78차원  (목표: 78)


In [8]:
# [07] 피처 표준화 및 텐서 변환  (훈련 데이터로만 fit — 데이터 누수 방지)
scaler     = StandardScaler()
scaler.fit(df_feat[NUM_COLS].values[idx_train])          # 훈련만 fit
num_matrix = scaler.transform(df_feat[NUM_COLS].values)  # 전체 transform
X_all      = np.hstack([num_matrix, text_embed]).astype(np.float32)
X_all      = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)
in_dim     = X_all.shape[1]

x_rev  = torch.tensor(X_all,   dtype=torch.float32)
y      = torch.tensor(labels,  dtype=torch.long)

print(f'✅ 피처 행렬: {x_rev.shape}')
print(f'   사기={y.sum().item():,}  정상={(y==0).sum().item():,}')

✅ 피처 행렬: torch.Size([50950, 78])
   사기=12,659  정상=38,291


In [9]:
# [08] 엣지 구성 (R-S-R + R-Sim-R)
rng = np.random.default_rng(42)

def make_pairs(groups, max_per_node=None):
    src_list, dst_list = [], []
    for nodes in groups:
        nodes = list(nodes)
        if len(nodes) < 2:
            continue
        n_pairs = len(nodes) * (len(nodes) - 1) // 2
        if max_per_node is None or n_pairs <= max_per_node * len(nodes):
            for i in range(len(nodes)):
                for j in range(i + 1, len(nodes)):
                    src_list.append(nodes[i]); dst_list.append(nodes[j])
        else:
            for node in nodes:
                others = [n for n in nodes if n != node]
                k = min(max_per_node, len(others))
                sampled = rng.choice(others, k, replace=False)
                for j in sampled:
                    src_list.append(node); dst_list.append(int(j))
    return np.array(src_list, dtype=np.int64), np.array(dst_list, dtype=np.int64)

def to_edge(src, dst, n_nodes):
    if len(src) == 0:
        return torch.zeros((2, 0), dtype=torch.long)
    ei = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    ei = to_undirected(ei, num_nodes=n_nodes)
    ei = ei[:, ei[0] != ei[1]]
    return ei

print(f'엣지 구성 시작...')
t0 = time.time()

# R-S-R: 같은 식당 + 같은 별점 (유저 신원 미사용 — 누수 없음)
grp_rsr  = df.groupby(['prod_id', 'rating']).apply(lambda x: x.index.tolist()).values
s, d     = make_pairs(grp_rsr, max_per_node=CFG['rsr_max_per_node'])
edge_rsr = to_edge(s, d, N)
print(f'  R-S-R  : {edge_rsr.size(1):>8,}개 엣지  (같은 식당+별점)')

# R-Sim-R: 텍스트 유사도 >= rsim_thr
sim_df   = pd.read_csv('custom_edges_textSim.csv')
sim_df   = sim_df[sim_df['text_similarity'] >= CFG['rsim_thr']].copy()
valid    = (sim_df['src_review_idx'] < N) & (sim_df['dst_review_idx'] < N)
sim_df   = sim_df[valid]
s        = sim_df['src_review_idx'].values.astype(np.int64)
d        = sim_df['dst_review_idx'].values.astype(np.int64)
edge_rsim = to_edge(s, d, N)
print(f'  R-Sim-R: {edge_rsim.size(1):>8,}개 엣지  (유사도 >= {CFG["rsim_thr"]})')

# 관계 타입 결합 (0=R-S-R, 1=R-Sim-R)
edge_index = torch.cat([edge_rsr, edge_rsim], dim=1)
edge_type  = torch.cat([
    torch.zeros(edge_rsr.size(1),  dtype=torch.long),
    torch.ones( edge_rsim.size(1), dtype=torch.long)
])
print(f'\n✅ 엣지 구성 완료 ({time.time()-t0:.0f}s) | 총 {edge_index.size(1):,}개')

엣지 구성 시작...
  R-S-R  :  926,748개 엣지  (같은 식당+별점)
  R-Sim-R:  319,670개 엣지  (유사도 >= 0.7)

✅ 엣지 구성 완료 (3s) | 총 1,246,418개


In [10]:
# [09] ReviewRGCN 모델 정의
class ReviewRGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_class=2, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_dim, hidden_dim, num_relations=2)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=2)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_class)
        )

    def forward(self, x, edge_index, edge_type):
        h = F.elu(self.norm1(self.conv1(x, edge_index, edge_type)))
        h = self.drop(h)
        h = F.elu(self.norm2(self.conv2(h, edge_index, edge_type)))
        h = self.drop(h)
        return self.clf(h)

m_test   = ReviewRGCN(in_dim=in_dim, hidden_dim=CFG['hidden_dim'])
n_params = sum(p.numel() for p in m_test.parameters() if p.requires_grad)
print(f'✅ ReviewRGCN 정의 완료 | 파라미터: {n_params:,}개')
del m_test

✅ ReviewRGCN 정의 완료 | 파라미터: 88,258개


In [11]:
# [10] 학습 유틸리티
def eval_model(model, x, ei, et, y, mask):
    model.eval()
    with torch.no_grad():
        logits = model(x, ei, et)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = logits.argmax(dim=1)
    yt    = y[mask].numpy()
    yp    = preds[mask].numpy()
    yprob = probs[mask].numpy()
    return average_precision_score(yt, yprob), f1_score(yt, yp, average='macro'), yt, yp, yprob

def train_step(model, opt, crit, x, ei, et, y, mask):
    model.train(); opt.zero_grad()
    logits = model(x, ei, et)
    loss   = crit(logits[mask], y[mask])
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    return float(loss)

def bar(v, L=28): return '█'*int(v*L) + '░'*(L-int(v*L))

cw   = compute_class_weight('balanced', classes=np.array([0,1]), y=labels[idx_train])
print(f'클래스 가중치: 정상={cw[0]:.3f}, 사기={cw[1]:.3f}')

클래스 가중치: 정상=0.654, 사기=2.124


In [12]:
# [11] ReviewRGCN 학습
print('='*56)
print(' ReviewRGCN 학습  (R-S-R + R-Sim-R | 78차원)')
print('='*56)

crit  = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32))
model = ReviewRGCN(in_dim=in_dim, hidden_dim=CFG['hidden_dim'], dropout=CFG['dropout'])
opt   = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
sch   = CosineAnnealingLR(opt, T_max=CFG['epochs'])

best_pr, best_f1, best_state = 0.0, 0.0, None
no_imp = 0
t0     = time.time()

for ep in range(1, CFG['epochs'] + 1):
    loss = train_step(model, opt, crit, x_rev, edge_index, edge_type, y, train_mask)
    sch.step()
    if ep % 5 == 0:
        vpr, vf1, *_ = eval_model(model, x_rev, edge_index, edge_type, y, val_mask)
        print(f'  [E{ep:3d}] loss={loss:.4f} | val PR-AUC={vpr:.4f} F1={vf1:.4f} ({time.time()-t0:.0f}s)')
        if vpr > best_pr:
            best_pr, best_f1 = vpr, vf1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= CFG['patience']:
                print(f'  Early stopping at epoch {ep}')
                break

model.load_state_dict(best_state)
_, _, yt, yp, yprob = eval_model(model, x_rev, edge_index, edge_type, y, test_mask)
res_pr = average_precision_score(yt, yprob)
res_f1 = f1_score(yt, yp, average='macro')
torch.save(best_state, 'best_review_rgcn.pt')
print(f'\n✅ ReviewRGCN 완료:  PR-AUC={res_pr:.4f}  Macro F1={res_f1:.4f}')

 ReviewRGCN 학습  (R-S-R + R-Sim-R | 78차원)
  [E  5] loss=0.5125 | val PR-AUC=0.6704 F1=0.6893 (10s)
  [E 10] loss=0.4098 | val PR-AUC=0.6619 F1=0.7604 (18s)
  [E 15] loss=0.3867 | val PR-AUC=0.6179 F1=0.7772 (26s)
  [E 20] loss=0.3757 | val PR-AUC=0.6017 F1=0.7632 (34s)
  [E 25] loss=0.3675 | val PR-AUC=0.5901 F1=0.7804 (41s)
  [E 30] loss=0.3582 | val PR-AUC=0.6144 F1=0.7790 (50s)
  [E 35] loss=0.3571 | val PR-AUC=0.6548 F1=0.7755 (58s)
  [E 40] loss=0.3531 | val PR-AUC=0.6814 F1=0.7794 (66s)
  [E 45] loss=0.3505 | val PR-AUC=0.6927 F1=0.7794 (75s)
  [E 50] loss=0.3469 | val PR-AUC=0.7020 F1=0.7801 (84s)
  [E 55] loss=0.3465 | val PR-AUC=0.7104 F1=0.7807 (93s)
  [E 60] loss=0.3459 | val PR-AUC=0.7167 F1=0.7810 (101s)
  [E 65] loss=0.3431 | val PR-AUC=0.7212 F1=0.7827 (109s)
  [E 70] loss=0.3424 | val PR-AUC=0.7243 F1=0.7831 (118s)
  [E 75] loss=0.3431 | val PR-AUC=0.7269 F1=0.7834 (126s)
  [E 80] loss=0.3432 | val PR-AUC=0.7286 F1=0.7831 (133s)
  [E 85] loss=0.3432 | val PR-AUC=0.7295 F

In [ ]:
# [12] 최종 리포트 + 결과 저장
W = 56
print()
print('='*W)
print('  04 Baseline — ReviewRGCN 결과')
print('='*W)
print(f'\n  노드 : Review 1종 (78차원)')
print(f'  엣지 : R-S-R + R-Sim-R ({edge_index.size(1):,}개)')
print()
print(f'  PR-AUC  : [{bar(res_pr)}] {res_pr:.4f}')
print(f'  Macro F1: [{bar(res_f1)}] {res_f1:.4f}')
print()
print('  [Classification Report — Test Set]')
print(classification_report(yt, yp, target_names=['정상(0)','사기(1)'], digits=4))

results = {
    'model'    : 'ReviewRGCN',
    'notebook' : '04_Baseline',
    'nodes'    : 'Review only (78-dim)',
    'relations': ['R-S-R', 'R-Sim-R'],
    'n_edges'  : int(edge_index.size(1)),
    'pr_auc'   : round(float(res_pr), 4),
    'macro_f1' : round(float(res_f1), 4),
    'split'    : 'temporal_option_b',
}
with open('results_04.json', 'w', encoding='utf-8') as f:
    _json.dump(results, f, ensure_ascii=False, indent=2)
torch.save(best_state, 'best_review_rgcn.pt')

print('='*W)
print('✅ 저장 완료: results_04.json / best_review_rgcn.pt')
print('다음 단계: 05_Model_Training.ipynb 에서 SixRelCAREGNN과 비교')


  04 Baseline — ReviewRGCN 결과

  노드 : Review 1종 (78차원)
  엣지 : R-S-R + R-Sim-R (1,246,418개)

  PR-AUC  : [████████████████░░░░░░░░░░░░] 0.6041
  Macro F1: [████████████████░░░░░░░░░░░░] 0.5896

  [Classification Report — Test Set]
              precision    recall  f1-score   support

       정상(0)     0.9605    0.4484    0.6114     10201
       사기(1)     0.4044    0.9531    0.5679      4009

    accuracy                         0.5908     14210
   macro avg     0.6825    0.7007    0.5896     14210
weighted avg     0.8036    0.5908    0.5991     14210

✅ 저장 완료: results_04.json / best_review_rgcn.pt
다음 단계: 05_Model_Training.ipynb 에서 SixRelCAREGNN과 비교


: 